# ⚡ PySpark on Olist — A Compact API-Fluency Demo

**Project:** E-Commerce Business Intelligence & SQL Analytics (portfolio project 26)  
**Dataset:** Olist Brazilian E-Commerce (Kaggle — `olistbr/brazilian-ecommerce`) · ~100K orders · 9 CSV files · Sept 2016 – Oct 2018  
**Goal:** Re-express the repo's core SQL KPIs in **PySpark** — DataFrame API *and* `spark.sql` — to show Spark fluency: explicit schemas, broadcast joins, window functions, caching, partitioned Parquet and reading `explain()` plans.  
**Author:** Narendra Kalam

---

## 📌 Read this first — honest scope
Olist is only ~ 100K orders (~ 120 MB of CSV). **Spark is not the right tool at this size** — pandas or SQLite finish the same work with less overhead, and Step 6 measures that on this very machine. This notebook is an **API-fluency demo on a small dataset**, not a claim that Spark was needed. The point is to show *how* the same logic is written in Spark and *when* Spark starts to matter.

## 🗺️ Contents
1. Setup — `SparkSession` (`local[*]`) + 9 CSVs with explicit schemas  
2. Joins — orders · items · payments · customers · products (+ sellers, category lookup) with `broadcast`  
3. KPIs — monthly revenue · late-delivery rate · top categories · repeat-customer rate  
4. Window functions — `rank` / `dense_rank` top sellers per state · running revenue total · the same in `spark.sql` with CTEs  
5. Optimisation — `explain()` · `cache()` · partitioned Parquet + partition pruning  
6. Honest check — Spark vs pandas on the same aggregate  

---
## Step 1 — Setup: SparkSession and 9 CSVs with explicit schemas
**Why:** `master("local[*]")` runs Spark inside this notebook using every local core — no cluster needed. Reading CSVs with `inferSchema` costs an extra full pass and can silently guess wrong types, so every table gets an explicit `StructType`, read in `FAILFAST` mode so a bad row stops the notebook instead of becoming a quiet `null`.

**Business insight:** Schema-on-read is where pipelines silently break in production. Declaring types up-front is the Spark equivalent of the repo's data-quality gate.

> **Environment:** needs Java (8/11/17 for Spark 3.5, 17/21 for Spark 4). On Kaggle, attach the Olist dataset and run — the data folder is auto-detected. If `SparkSession` complains about the Java version, install a matching `pyspark` (e.g. `pip install pyspark==3.5.5`).

In [1]:
import os, sys, time, glob, shutil, subprocess
from pathlib import Path

try:
    import pyspark
except ImportError:                                   # e.g. fresh Kaggle kernel
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
    import pyspark

import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.functions import broadcast
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               DoubleType, TimestampType)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

OUT_DIR = Path("pyspark_outputs"); OUT_DIR.mkdir(exist_ok=True)   # scratch space (git-ignored)

t0 = time.perf_counter()
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("olist-pyspark-demo")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")          # default 200 is overkill for ~100K rows
    .config("spark.sql.session.timeZone", "UTC")          # avoid DST gaps when parsing 2016-18 timestamps
    .config("spark.sql.warehouse.dir", str((OUT_DIR / "spark-warehouse").resolve()))
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.maxMetadataStringLength", "250")   # keep explain() lines from being cut with "..."
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
SPARK_STARTUP_S = time.perf_counter() - t0
print(f"Spark {spark.version} | master={spark.sparkContext.master} | "
      f"default parallelism={spark.sparkContext.defaultParallelism} | startup={SPARK_STARTUP_S:.1f}s")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/23 08:47:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.4 | master=local[*] | default parallelism=4 | startup=10.5s


**📌 Observation:** Spark started in local mode and uses every core visible to the notebook (the parallelism figure above is that core count).

**💡 Pipeline / Engineering decision:** `shuffle.partitions` is lowered from Spark's default of 200 to 8 — with ~100K rows, 200 tiny shuffle tasks would be pure overhead. The session time zone is pinned to UTC so timestamp parsing is identical on a laptop, in Kaggle and on a server.

In [2]:
def find_data_dir() -> Path:
    """Env var -> Kaggle input -> local folders (full data) -> repo sample."""
    candidates = []
    if os.environ.get("RAW_DATA_DIR"):
        candidates.append(Path(os.environ["RAW_DATA_DIR"]))
    candidates += [Path(p).parent for p in
                   glob.glob("/kaggle/input/**/olist_orders_dataset.csv", recursive=True)]
    candidates += [Path("D:/Data Science Datasets/Olist Brazilian E-Commerce"),
                   Path("../data/raw"), Path("data/raw"),
                   Path("../data/sample"), Path("data/sample")]
    for c in candidates:
        if (c / "olist_orders_dataset.csv").exists():
            return c
    raise FileNotFoundError("Olist CSVs not found - attach the Kaggle dataset or set RAW_DATA_DIR")

DATA_DIR = find_data_dir()

TS, S, I, D = TimestampType(), StringType(), IntegerType(), DoubleType()

FILES = {
    "customers":            "olist_customers_dataset.csv",
    "orders":               "olist_orders_dataset.csv",
    "order_items":          "olist_order_items_dataset.csv",
    "payments":             "olist_order_payments_dataset.csv",
    "reviews":              "olist_order_reviews_dataset.csv",
    "products":             "olist_products_dataset.csv",
    "sellers":              "olist_sellers_dataset.csv",
    "geolocation":          "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

SCHEMAS = {
    "customers": StructType([
        StructField("customer_id", S), StructField("customer_unique_id", S),
        StructField("customer_zip_code_prefix", I), StructField("customer_city", S),
        StructField("customer_state", S)]),
    "orders": StructType([
        StructField("order_id", S), StructField("customer_id", S), StructField("order_status", S),
        StructField("order_purchase_timestamp", TS), StructField("order_approved_at", TS),
        StructField("order_delivered_carrier_date", TS), StructField("order_delivered_customer_date", TS),
        StructField("order_estimated_delivery_date", TS)]),
    "order_items": StructType([
        StructField("order_id", S), StructField("order_item_id", I), StructField("product_id", S),
        StructField("seller_id", S), StructField("shipping_limit_date", TS),
        StructField("price", D), StructField("freight_value", D)]),
    "payments": StructType([
        StructField("order_id", S), StructField("payment_sequential", I), StructField("payment_type", S),
        StructField("payment_installments", I), StructField("payment_value", D)]),
    "reviews": StructType([
        StructField("review_id", S), StructField("order_id", S), StructField("review_score", I),
        StructField("review_comment_title", S), StructField("review_comment_message", S),
        StructField("review_creation_date", TS), StructField("review_answer_timestamp", TS)]),
    "products": StructType([
        StructField("product_id", S), StructField("product_category_name", S),
        StructField("product_name_lenght", D), StructField("product_description_lenght", D),
        StructField("product_photos_qty", D), StructField("product_weight_g", D),
        StructField("product_length_cm", D), StructField("product_height_cm", D),
        StructField("product_width_cm", D)]),
    "sellers": StructType([
        StructField("seller_id", S), StructField("seller_zip_code_prefix", I),
        StructField("seller_city", S), StructField("seller_state", S)]),
    "geolocation": StructType([
        StructField("geolocation_zip_code_prefix", I), StructField("geolocation_lat", D),
        StructField("geolocation_lng", D), StructField("geolocation_city", S),
        StructField("geolocation_state", S)]),
    "category_translation": StructType([
        StructField("product_category_name", S), StructField("product_category_name_english", S)]),
}

def read_csv(name: str):
    path = DATA_DIR / FILES[name]
    if not path.exists():
        if name == "geolocation":                       # optional: not shipped in the repo's data/sample
            print("  (geolocation.csv not found - using an empty DataFrame with the right schema)")
            return spark.createDataFrame([], SCHEMAS[name])
        raise FileNotFoundError(path)
    return (spark.read
            .option("header", True)
            .option("mode", "FAILFAST")                  # bad row => error, never a silent null
            .option("multiLine", name == "reviews")      # review comments contain embedded newlines
            .option("quote", '"').option("escape", '"')
            .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
            .schema(SCHEMAS[name])                       # explicit schema - no inferSchema pass
            .csv(str(path)))

tables = {name: read_csv(name) for name in FILES}
customers, orders, items, payments = (tables[k] for k in ("customers", "orders", "order_items", "payments"))
reviews, products, sellers = (tables[k] for k in ("reviews", "products", "sellers"))
geolocation, category_translation = tables["geolocation"], tables["category_translation"]

inventory = pd.DataFrame({
    "table": list(tables),
    "rows": [df.count() for df in tables.values()],
    "columns": [len(df.columns) for df in tables.values()],
})
display(inventory)

,table,rows,columns
0,customers,99441,5
1,orders,99441,8
2,order_items,112650,7
3,payments,103886,5
4,reviews,99224,7
5,products,32951,9
6,sellers,3095,4
7,geolocation,1000163,5
8,category_translation,71,2


**📌 Observation:** All 9 files load under `FAILFAST` with explicit schemas and the row counts match the published Olist dataset (99,441 orders · 112,650 items · 103,886 payments · 99,224 reviews · 32,951 products · 3,095 sellers · 1,000,163 geolocation rows · 71 category translations).

**💡 Pipeline / Engineering decision:** `geolocation` (the largest file) is loaded to complete the 9-table inventory but is not used by any KPI — Spark's lazy evaluation means it costs nothing beyond the one `count()` above. `reviews` needs `multiLine=True` because comments contain line breaks.

---
## Step 2 — Joins: orders · items · payments · customers · products
**Why:** Grain decides correctness. `order_items` is one row per *item*, `payments` is one row per *payment instalment* — joining both straight onto orders multiplies rows and inflates revenue. So we aggregate each many-side table to **order grain first**, then join. Small dimension tables (`customers`, `products`, `sellers`, `category_translation`) are wrapped in `broadcast()` so Spark ships them to every task instead of shuffling the big tables.

**Business insight:** A fan-out bug here would overstate revenue in every downstream KPI. The reconciliation check below proves the two fact tables agree and shows how large the naive-join error would have been.

In [3]:
delivered = orders.filter(F.col("order_status") == "delivered")      # same scope as sql/*.sql

# --- many-side tables -> order grain BEFORE joining (prevents fan-out) -------------
order_items_agg = items.groupBy("order_id").agg(
    F.sum(F.col("price") + F.col("freight_value")).alias("gross_revenue"),
    F.count("*").alias("n_items"),
)
order_pay_agg = payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("paid_value"),
    F.max("payment_installments").alias("max_installments"),
)

# --- order-grain fact: one row per delivered order ---------------------------------
orders_fact = (
    delivered
    .join(order_items_agg, "order_id", "inner")
    .join(order_pay_agg, "order_id", "left")
    .join(broadcast(customers), "customer_id", "inner")              # small dim -> broadcast
    .withColumn("purchase_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
)

# --- item-grain fact: one row per delivered order item -----------------------------
def build_item_fact():
    return (
        items
        .join(delivered.select("order_id", "customer_id", "order_purchase_timestamp"), "order_id")
        .join(broadcast(customers.select("customer_id", "customer_unique_id", "customer_state")), "customer_id")
        .join(broadcast(products.select("product_id", "product_category_name")), "product_id", "left")
        .join(broadcast(category_translation), "product_category_name", "left")
        .join(broadcast(sellers.select("seller_id", "seller_state")), "seller_id", "left")
        .withColumn("category", F.coalesce("product_category_name_english", "product_category_name", F.lit("unknown")))
        .withColumn("revenue", F.col("price") + F.col("freight_value"))
        .withColumn("purchase_year", F.year("order_purchase_timestamp"))
        .withColumn("purchase_month_num", F.month("order_purchase_timestamp"))
        .withColumn("purchase_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    )

item_fact = build_item_fact()

print("orders_fact columns:", orders_fact.columns)
print("item_fact   columns:", item_fact.columns)

orders_fact columns: ['customer_id', 'order_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'gross_revenue', 'n_items', 'paid_value', 'max_installments', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'purchase_month']
item_fact   columns: ['seller_id', 'product_category_name', 'product_id', 'customer_id', 'order_id', 'order_item_id', 'shipping_limit_date', 'price', 'freight_value', 'order_purchase_timestamp', 'customer_unique_id', 'customer_state', 'product_category_name_english', 'seller_state', 'category', 'revenue', 'purchase_year', 'purchase_month_num', 'purchase_month']


In [4]:
# Reconciliation: both facts must give the same revenue; a naive items x payments join must not.
rev_item_grain  = item_fact.agg(F.sum("revenue")).first()[0]
rev_order_grain = orders_fact.agg(F.sum("gross_revenue")).first()[0]
rev_naive = (items.join(delivered.select("order_id"), "order_id")
                  .join(payments, "order_id")                       # fan-out: items x payment rows
                  .agg(F.sum(F.col("price") + F.col("freight_value"))).first()[0])

print(f"Revenue, item-grain fact   : R$ {rev_item_grain:>14,.2f}")
print(f"Revenue, order-grain fact  : R$ {rev_order_grain:>14,.2f}")
print(f"Revenue, NAIVE items x pay : R$ {rev_naive:>14,.2f}   (+{100*(rev_naive/rev_item_grain-1):.2f}% inflated)")
assert abs(rev_item_grain - rev_order_grain) < 0.01, "fact tables disagree"

Revenue, item-grain fact   : R$  15,419,773.75
Revenue, order-grain fact  : R$  15,419,773.75
Revenue, NAIVE items x pay : R$  16,114,019.06   (+4.50% inflated)


**📌 Observation:** Item-grain and order-grain facts agree to the cent (R$ 15,419,773.75), while the naive `items × payments` join inflates revenue by **4.50%** — orders paid in several instalments, or with several items, get counted more than once.

**💡 Pipeline / Engineering decision:** Aggregate the many-side tables to order grain *before* joining, and keep this reconciliation assert in the notebook — a silent fan-out is exactly the kind of bug that survives code review because the numbers still look plausible.

---
## Step 3 — KPIs: monthly revenue · late-delivery rate · top categories · repeat-customer rate
**Why:** These are the same four KPIs the SQLite layer in `sql/` computes — rebuilt with the DataFrame API. Definitions are kept identical (delivered orders only; revenue = price + freight; late = delivered after the estimated date; repeat customers counted on `customer_unique_id`, not `customer_id`) so the numbers are directly comparable with the rest of the repo.

**Business insight:** Finance wants the revenue trend, ops wants the SLA breach rate, merchandising wants category mix, growth wants the repeat rate — one Spark job each, all from the two fact tables above.

In [5]:
# 3.1 Monthly revenue, orders, AOV
monthly = (
    orders_fact.groupBy("purchase_month")
    .agg(F.count("order_id").alias("orders"), F.sum("gross_revenue").alias("gross_revenue"))
    .withColumn("avg_order_value", F.col("gross_revenue") / F.col("orders"))
    .orderBy("purchase_month")
)
monthly_pd = monthly.toPandas()
display(monthly_pd.head(8)); print(f"... {len(monthly_pd)} months in total")

,purchase_month,orders,gross_revenue,avg_order_value
0,2016-09,1,143.46,143.46
1,2016-10,265,"46,490.66",175.44
2,2016-12,1,19.62,19.62
3,2017-01,750,"127,482.37",169.98
4,2017-02,1653,"271,239.32",164.09
5,2017-03,2546,"414,330.95",162.74
6,2017-04,2303,"390,812.40",169.70
7,2017-05,3546,"566,851.40",159.86


... 23 months in total


**📌 Observation:** 23 months of delivered-order revenue (Sep 2016 – Aug 2018). The first months are tiny (one order in Sep 2016, one in Dec 2016, none in Nov 2016), then volume ramps steeply through 2017 — the same 2016 ramp-up the SQL layer's anomaly monitor flags.

**💡 Pipeline / Engineering decision:** `toPandas()` is used only on tiny, already-aggregated results (23 rows); collecting a large DataFrame to the driver would defeat the purpose of Spark.

In [6]:
# 3.2 Late-delivery rate (delivered after the estimated date)
late = (
    delivered.filter(F.col("order_delivered_customer_date").isNotNull()
                     & F.col("order_estimated_delivery_date").isNotNull())
    .agg(F.count("*").alias("delivered_orders"),
         F.sum(F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1)
                .otherwise(0)).alias("late_orders"))
    .withColumn("late_pct", F.round(100 * F.col("late_orders") / F.col("delivered_orders"), 2))
)
display(late.toPandas())

# 3.3 Top 10 categories by product revenue (English names)
top_categories = (
    item_fact.groupBy("category")
    .agg(F.countDistinct("order_id").alias("orders"),
         F.sum("price").alias("product_revenue"),
         F.avg("price").alias("avg_item_price"))
    .orderBy(F.desc("product_revenue")).limit(10)
)
display(top_categories.toPandas())

,delivered_orders,late_orders,late_pct
0,96470,7826,8.11


,category,orders,product_revenue,avg_item_price
0,health_beauty,8647,"1,233,131.72",130.28
1,watches_gifts,5495,"1,166,176.98",199.04
2,bed_bath_table,9272,"1,023,434.76",93.44
3,sports_leisure,7530,"954,852.55",113.25
4,computers_accessories,6530,"888,724.61",116.26
5,furniture_decor,6307,"711,927.69",87.25
6,housewares,5743,"615,628.69",90.60
7,cool_stuff,3559,"610,204.10",164.12
8,auto,3810,"578,966.65",139.85
9,toys,3804,"471,286.48",116.94


**📌 Observation:** The late-delivery rate is **8.11%** (7,826 of 96,470 delivered orders that carry both a delivered and an estimated date — 8 delivered orders have no delivery timestamp, so they are excluded from this KPI exactly as in `sql/03_delivery_sla.sql`). `health_beauty` leads categories by product revenue, followed by `watches_gifts` and `bed_bath_table`.

**💡 Business insight:** `bed_bath_table` has far more orders than `watches_gifts` but lower revenue — average item price is under half (R$ 93 vs R$ 199). Revenue ranking and order-volume ranking tell different stories, so merchandising should look at both.

In [7]:
# 3.4 Repeat-customer rate - grain is customer_unique_id (one person can own several customer_ids)
customer_orders = orders_fact.groupBy("customer_unique_id").agg(F.countDistinct("order_id").alias("n_orders"))
repeat = (
    customer_orders.agg(F.count("*").alias("customers"),
                        F.sum(F.when(F.col("n_orders") > 1, 1).otherwise(0)).alias("repeat_customers"))
    .withColumn("repeat_rate_pct", F.round(100 * F.col("repeat_customers") / F.col("customers"), 2))
)
display(repeat.toPandas())

# ---- KPI scoreboard (one row, comparable with the repo's SQL pipeline) -------------
tot = orders_fact.agg(F.sum("gross_revenue").alias("rev"), F.count("*").alias("orders")).first()
late_row, rep_row = late.first(), repeat.first()
scoreboard = pd.DataFrame({
    "kpi": ["total_revenue", "delivered_orders", "avg_order_value", "late_delivery_pct", "repeat_purchase_pct"],
    "pyspark": [round(tot["rev"], 2), tot["orders"], round(tot["rev"] / tot["orders"], 2),
                late_row["late_pct"], rep_row["repeat_rate_pct"]],
})
# Reference = artifacts/kpi_snapshot.json produced by the SQLite pipeline on the FULL dataset (see README)
SQL_REFERENCE = {"total_revenue": 15419773.75, "delivered_orders": 96478, "avg_order_value": 159.83,
                 "late_delivery_pct": 8.11, "repeat_purchase_pct": 3.0}
if inventory.set_index("table").loc["orders", "rows"] > 90_000:      # only meaningful on the full dataset
    scoreboard["sql_pipeline"] = scoreboard["kpi"].map(SQL_REFERENCE)
    scoreboard["abs_diff"] = (scoreboard["pyspark"] - scoreboard["sql_pipeline"]).abs()
else:
    print("Sample dataset detected - comparison with the full-dataset SQL reference skipped.")
display(scoreboard)

,customers,repeat_customers,repeat_rate_pct
0,93358,2801,3.00


,kpi,pyspark,sql_pipeline,abs_diff
0,total_revenue,"15,419,773.75","15,419,773.75",0.00
1,delivered_orders,"96,478.00","96,478.00",0.00
2,avg_order_value,159.83,159.83,0.00
3,late_delivery_pct,8.11,8.11,0.00
4,repeat_purchase_pct,3.00,3.00,0.00


**📌 Observation:** Repeat-customer rate is **3.00%** (2,801 of 93,358 unique customers), and all five scoreboard KPIs match the SQLite pipeline's snapshot with a difference of 0.00 — the PySpark rewrite reproduces the SQL layer exactly.

**💡 Business insight:** Counting on `customer_unique_id` matters: Olist issues a new `customer_id` per order, so counting on `customer_id` would report ~0% repeat buyers. The low repeat rate is a genuine dataset characteristic, not a bug.

---
## Step 4 — Window functions: top sellers per state and running revenue
**Why:** Window functions compute a value *per row* using its neighbours, without collapsing rows like `groupBy`. `Window.partitionBy("seller_state").orderBy(desc("revenue"))` restarts the ranking inside every state. `rank` leaves gaps after ties (1, 2, 2, 4); `dense_rank` does not (1, 2, 2, 3). A running total is a `sum` over a frame that grows from the first month to the current one.

**Business insight:** "Who are the top sellers in each state?" and "how fast is cumulative revenue building?" are the two window-function questions analysts get asked most. Sellers need at least `MIN_ORDERS_FOR_SELLER_RANK` orders (same rule as `src/config.py`) so one lucky order does not make a top seller.

In [8]:
MIN_ORDERS = 5            # = MIN_ORDERS_FOR_SELLER_RANK in src/config.py

seller_perf = (
    item_fact.groupBy("seller_id", "seller_state")
    .agg(F.countDistinct("order_id").alias("orders"), F.sum("revenue").alias("revenue"))
    .filter(F.col("orders") >= MIN_ORDERS)
)

# 4.1 Top 3 sellers per state by revenue: rank vs dense_rank
w_rev = Window.partitionBy("seller_state").orderBy(F.desc("revenue"))
top_sellers = (
    seller_perf
    .withColumn("rank", F.rank().over(w_rev))
    .withColumn("dense_rank", F.dense_rank().over(w_rev))
    .filter(F.col("rank") <= 3)
    .orderBy("seller_state", "rank")
)
display(top_sellers.limit(12).toPandas())

# Ties are rare for revenue (floats) but common for order counts - so show the rank/dense_rank gap there
w_ord = Window.partitionBy("seller_state").orderBy(F.desc("orders"))
ties_demo = (
    seller_perf.filter(F.col("seller_state") == "SP")
    .withColumn("rank_by_orders", F.rank().over(w_ord))
    .withColumn("dense_rank_by_orders", F.dense_rank().over(w_ord))
    .filter(F.col("rank_by_orders") != F.col("dense_rank_by_orders"))
    .select("seller_id", "orders", "rank_by_orders", "dense_rank_by_orders")
    .orderBy("rank_by_orders").limit(6)
)
print("rank vs dense_rank after ties (state SP, ranked by order count):")
display(ties_demo.toPandas())

,seller_id,seller_state,orders,revenue,rank,dense_rank
0,53243585a1d6dc2643021fd1853d8905,BA,348,"230,797.02",1,1
1,c72de06d72748d1a0dfb2125be43ba63,BA,21,"18,733.18",2,2
2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,57,"13,969.77",3,3
3,bbf9ad41dca6603e614efcdad7aab8c4,CE,14,"8,622.26",1,1
4,dbdd0ec73a4817971d962698f2fea022,CE,16,"8,174.80",2,2
5,8d79c8a04e42d722a75097ce5cbcf2ef,CE,26,"3,347.25",3,3
6,44073f8b7e41514de3b7815dd0237f4f,DF,142,"21,519.49",1,1
7,f3b80352b986ab4d1057a4b724be19d0,DF,86,"11,941.36",2,2
8,5656537e588803a555b8eb41f07a944b,DF,171,"11,255.55",3,3
9,001cca7ae9ae17fb1caed9dfb1094831,ES,195,"33,142.90",1,1


rank vs dense_rank after ties (state SP, ranked by order count):


,seller_id,orders,rank_by_orders,dense_rank_by_orders
0,4d6d651bd7684af3fffabd5f08d12e5a,363,32,31
1,c826c40d7b19f62a09e2d7c5e7295ee2,346,33,32
2,92eb0f42c21942b6552362b9b114707d,334,34,33
3,7ddcbb64b5bc1ef36ca8c151f6ec77df,325,35,34
4,b2ba3715d723d245138f291a6fe42594,324,36,35
5,5dceca129747e92ff8ef7a997dc4f8ca,322,37,36


**📌 Observation:** Revenue is heavily concentrated at the top of each state — e.g. the #1 seller in BA earns about R$ 231K from 348 orders, over 12× the #2 seller. `rank` and `dense_rank` are identical for revenue (no ties), but on order counts they diverge once ties appear: the seller shown at `rank` 32 has `dense_rank` 31 because `rank` skips positions after tied sellers.

**💡 Pipeline / Engineering decision:** Use `dense_rank` when you want "top N *distinct* levels", `rank` when you want standard competition ranking, and `row_number` when you need exactly N rows no matter what.

In [9]:
# 4.2 Running (cumulative) revenue total over months
w_run = Window.orderBy("purchase_month").rowsBetween(Window.unboundedPreceding, Window.currentRow)
running = (
    monthly.select("purchase_month", "gross_revenue")
    .withColumn("running_revenue", F.sum("gross_revenue").over(w_run))
)
display(running.toPandas().tail(6))

,purchase_month,gross_revenue,running_revenue
17,2018-03,"1,120,598.24","10,132,843.09"
18,2018-04,"1,132,878.93","11,265,722.02"
19,2018-05,"1,128,774.52","12,394,496.54"
20,2018-06,"1,011,978.29","13,406,474.83"
21,2018-07,"1,027,807.28","14,434,282.11"
22,2018-08,"985,491.64","15,419,773.75"


**📌 Observation:** The final running total (R$ 15,419,773.75) equals total revenue from Step 2 — a cheap built-in sanity check for any cumulative window.

**💡 Pipeline / Engineering decision:** A window with `orderBy` but no `partitionBy` moves all rows into one partition. That is fine here because it runs on 23 already-aggregated monthly rows; on raw event-level data you would partition (e.g. by year) or aggregate first.

### 4.3 The same logic in `spark.sql` with CTEs
**Why:** Spark SQL and the DataFrame API compile to the *same* optimised plan, so the choice is readability. Registering the fact tables as temp views lets the repo's SQL skills (CTEs, `RANK() OVER`, frame clauses) run unchanged on Spark — and the equality check proves both styles agree.

In [10]:
item_fact.createOrReplaceTempView("item_fact")
orders_fact.createOrReplaceTempView("orders_fact")

top_sellers_sql = spark.sql(f"""
    WITH seller_perf AS (
        SELECT seller_id, seller_state,
               COUNT(DISTINCT order_id) AS orders,
               SUM(revenue)             AS revenue
        FROM item_fact
        GROUP BY seller_id, seller_state
        HAVING COUNT(DISTINCT order_id) >= {MIN_ORDERS}
    ),
    ranked AS (
        SELECT *,
               RANK()       OVER (PARTITION BY seller_state ORDER BY revenue DESC) AS rank,
               DENSE_RANK() OVER (PARTITION BY seller_state ORDER BY revenue DESC) AS dense_rank
        FROM seller_perf
    )
    SELECT seller_id, seller_state, orders, revenue, rank, dense_rank
    FROM ranked
    WHERE rank <= 3
    ORDER BY seller_state, rank
""")

running_sql = spark.sql("""
    WITH monthly AS (
        SELECT purchase_month, SUM(gross_revenue) AS gross_revenue
        FROM orders_fact
        GROUP BY purchase_month
    )
    SELECT purchase_month, gross_revenue,
           SUM(gross_revenue) OVER (ORDER BY purchase_month
                                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue
    FROM monthly
    ORDER BY purchase_month
""")

def same_rows(a, b, cols):
    """True when both DataFrames hold identical rows (float columns compared at 2 dp)."""
    norm = lambda df: df.select([F.round(c, 2).alias(c) if t == "double" else F.col(c)
                                 for c, t in df.select(cols).dtypes])
    a, b = norm(a.select(cols)), norm(b.select(cols))
    return a.exceptAll(b).count() == 0 and b.exceptAll(a).count() == 0

print("top sellers   - DataFrame API == spark.sql:",
      same_rows(top_sellers, top_sellers_sql, ["seller_id", "seller_state", "orders", "revenue", "rank", "dense_rank"]))
print("running total - DataFrame API == spark.sql:",
      same_rows(running, running_sql, ["purchase_month", "gross_revenue", "running_revenue"]))
display(top_sellers_sql.limit(6).toPandas())

top sellers   - DataFrame API == spark.sql: True
running total - DataFrame API == spark.sql: True


,seller_id,seller_state,orders,revenue,rank,dense_rank
0,53243585a1d6dc2643021fd1853d8905,BA,348,"230,797.02",1,1
1,c72de06d72748d1a0dfb2125be43ba63,BA,21,"18,733.18",2,2
2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,57,"13,969.77",3,3
3,bbf9ad41dca6603e614efcdad7aab8c4,CE,14,"8,622.26",1,1
4,dbdd0ec73a4817971d962698f2fea022,CE,16,"8,174.80",2,2
5,8d79c8a04e42d722a75097ce5cbcf2ef,CE,26,"3,347.25",3,3


**📌 Observation:** Both checks print `True` — the DataFrame API and `spark.sql` return identical rows. They go through the same Catalyst optimiser, so there is no performance reason to prefer one; choose whichever the team reads more easily.

**💡 Business insight:** The CTE + window SQL above is deliberately written in the same style as `sql/02_seller_performance.sql` and `sql/01_revenue_kpis.sql`, so the SQL skills built in this project transfer directly to Spark SQL, Databricks and BigQuery.

---
## Step 5 — Optimisation: `explain()`, `cache()`, partitioned Parquet
**Why:** Spark is lazy — nothing runs until an action — so performance work is mostly about *reading the plan* and *not recomputing*. We (a) check the physical plan really uses broadcast joins, (b) cache the item-grain fact that several KPIs reuse, and (c) write it as Parquet partitioned by year and month so a later query for one period reads only those folders.

**Business insight:** On a cluster these three habits (broadcast small tables, cache reused data, partition by the column you filter on) are what separate a 2-minute job from a 2-hour one. Here they are demonstrated, not needed.

### 5.1 Read the physical plan — `explain()`

In [11]:
# Built fresh from the raw CSVs (before anything is cached) so the plan shows the real joins
build_item_fact().groupBy("category").agg(F.sum("price").alias("product_revenue")).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[category#3566], functions=[sum(price#18)])
   +- Exchange hashpartitioning(category#3566, 8), ENSURE_REQUIREMENTS, [plan_id=20251]
      +- HashAggregate(keys=[category#3566], functions=[partial_sum(price#18)])
         +- Project [price#18, coalesce(product_category_name_english#51, product_category_name#33, unknown) AS category#3566]
            +- BroadcastHashJoin [seller_id#16], [seller_id#41], LeftOuter, BuildRight, false
               :- Project [product_category_name#33, seller_id#16, price#18, product_category_name_english#51]
               :  +- BroadcastHashJoin [product_category_name#33], [product_category_name#50], LeftOuter, BuildRight, false
               :     :- Project [seller_id#16, price#18, product_category_name#33]
               :     :  +- BroadcastHashJoin [product_id#15], [product_id#32], LeftOuter, BuildRight, false
               :     :     :- Project [product_id#15, seller_id

**📌 Observation (read the plan bottom-up):** each CSV is read with only the needed columns (`ReadSchema`) and the `order_status = delivered` filter is **pushed down** to the scan. All five joins are `BroadcastHashJoin` fed by a `BroadcastExchange` — there is **no `SortMergeJoin`** — and the plan contains a single shuffle (`Exchange hashpartitioning(category)`) for the final group-by. `AdaptiveSparkPlan isFinalPlan=false` means Spark's adaptive query execution may still refine the plan at run time.

**💡 Pipeline / Engineering decision:** Spark chose to broadcast the filtered `orders` side on its own as well; the explicit `broadcast()` hints on the dimension tables make that intent visible in the code and robust if table sizes change.

### 5.2 `cache()` — compute once, reuse many times

In [12]:
def run_kpis():
    """Three different aggregations that all reuse item_fact."""
    item_fact.groupBy("category").agg(F.sum("revenue")).count()
    item_fact.groupBy("seller_state").agg(F.countDistinct("order_id")).count()
    item_fact.groupBy("purchase_month").agg(F.avg("price")).count()

def best_of(fn, n=3):
    times = []
    for _ in range(n):
        t = time.perf_counter(); fn(); times.append(time.perf_counter() - t)
    return times

uncached = best_of(run_kpis)

item_fact.cache()
t = time.perf_counter(); n_rows = item_fact.count(); materialise_s = time.perf_counter() - t   # action fills the cache
cached = best_of(run_kpis)

print(f"item_fact rows cached : {n_rows:,}  (storage level: {item_fact.storageLevel})")
print(f"uncached runs (s)     : {[round(x, 2) for x in uncached]}")
print(f"cache materialise (s) : {materialise_s:.2f}")
print(f"cached runs (s)       : {[round(x, 2) for x in cached]}")

item_fact rows cached : 110,197  (storage level: Disk Memory Deserialized 1x Replicated)
uncached runs (s)     : [3.61, 3.03, 2.76]
cache materialise (s) : 2.62
cached runs (s)       : [0.77, 0.5, 0.4]


**📌 Observation:** The first call to `count()` after `cache()` pays a one-off cost to fill memory; every later aggregation then reads the in-memory copy instead of re-reading and re-joining the CSVs, and the cached runs are several times faster than the uncached ones (exact timings above depend on the machine).

**💡 Pipeline / Engineering decision:** Cache only data that is reused by several actions, and `unpersist()` it when done (Step 6 does this before benchmarking). Caching something used once only adds overhead.

### 5.3 Partitioned Parquet write

In [13]:
PARQUET_DIR = OUT_DIR / "item_fact_parquet"
shutil.rmtree(PARQUET_DIR, ignore_errors=True)

(item_fact
    .select("order_id", "order_item_id", "seller_id", "seller_state", "customer_state", "category",
            "price", "freight_value", "revenue", "purchase_year", "purchase_month_num")
    .repartition("purchase_year", "purchase_month_num")          # one file per partition folder
    .write.mode("overwrite")
    .partitionBy("purchase_year", "purchase_month_num")
    .parquet(str(PARQUET_DIR)))

files = [p for p in PARQUET_DIR.rglob("*.parquet")]
parquet_mb = sum(p.stat().st_size for p in files) / 1e6
csv_mb = (DATA_DIR / FILES["order_items"]).stat().st_size / 1e6
folders = sorted(p.relative_to(PARQUET_DIR).as_posix() for p in PARQUET_DIR.glob("purchase_year=*/purchase_month_num=*"))
print(f"{len(folders)} partition folders, {len(files)} parquet files, {parquet_mb:.1f} MB "
      f"(source order_items CSV: {csv_mb:.1f} MB, and the Parquet also holds joined columns)")
print("e.g.", folders[:3], "...", folders[-2:])

23 partition folders, 23 parquet files, 5.3 MB (source order_items CSV: 15.4 MB, and the Parquet also holds joined columns)
e.g. ['purchase_year=2016/purchase_month_num=10', 'purchase_year=2016/purchase_month_num=12', 'purchase_year=2016/purchase_month_num=9'] ... ['purchase_year=2018/purchase_month_num=7', 'purchase_year=2018/purchase_month_num=8']


**📌 Observation:** The write produced one folder per year/month (23 partitions, one Parquet file each) and the columnar, compressed Parquet is much smaller than the source `order_items` CSV even though it carries extra joined columns.

**💡 Pipeline / Engineering decision:** Partition by columns you filter on and that have *moderate* cardinality — year/month gives 23 folders here; partitioning by `seller_id` (3K values) would create thousands of tiny files. `repartition(...)` before the write keeps it to one file per folder.

### 5.4 Partition pruning — `explain()` on the Parquet read

In [14]:
q = (spark.read.parquet(str(PARQUET_DIR))
          .filter((F.col("purchase_year") == 2018) & (F.col("purchase_month_num") <= 3)))
q.explain()
display(q.agg(F.count("*").alias("items"), F.sum("revenue").alias("revenue")).toPandas())

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [order_id#9527,order_item_id#9528,seller_id#9529,seller_state#9530,customer_state#9531,category#9532,price#9533,freight_value#9534,revenue#9535,purchase_year#9536,purchase_month_num#9537] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/kaggle/working/pyspark_outputs/item_fact_parquet], PartitionFilters: [isnotnull(purchase_year#9536), isnotnull(purchase_month_num#9537), (purchase_year#9536 = 2018), (purchase_month_num#9537 <= 3)], PushedFilters: [], ReadSchema: struct<order_id:string,order_item_id:int,seller_id:string,seller_state:string,customer_state:string,category:string,price:double,freight_value:double,revenue:double>




,items,revenue
0,23572,"3,164,654.11"


**📌 Observation:** `PartitionFilters` shows the year/month predicates were applied to the **folder names**, so Spark reads only the 2018-01 to 2018-03 folders (3 of 23) and never opens the rest — result: 23,572 items and R$ 3,164,654.11 revenue for Jan–Mar 2018.

**💡 Business insight:** This is partition pruning — the reason lakehouse tables are laid out by date. A dashboard query for "last quarter" scans a few folders instead of the whole history, which is where Spark and Parquet start to earn their keep as data grows.

---
## Step 6 — Honest check: is Spark faster than pandas here?
**Why:** The claim in the intro should be measured, not asserted. Below, the *same* aggregate (delivered-order monthly revenue: read two CSVs → filter → join → group) runs in pandas and in Spark on this machine; results are compared first so the timing is apples-to-apples. Each engine is timed best-of-3 after the cache from Step 5 is released, and Spark's one-off JVM start-up time (Step 1) is reported separately.

In [15]:
item_fact.unpersist()

def pandas_monthly():
    o = pd.read_csv(DATA_DIR / FILES["orders"],
                    usecols=["order_id", "order_status", "order_purchase_timestamp"],
                    parse_dates=["order_purchase_timestamp"])
    i = pd.read_csv(DATA_DIR / FILES["order_items"], usecols=["order_id", "price", "freight_value"])
    m = o[o["order_status"] == "delivered"].merge(i, on="order_id")
    m["purchase_month"] = m["order_purchase_timestamp"].dt.strftime("%Y-%m")
    m["revenue"] = m["price"] + m["freight_value"]
    return (m.groupby("purchase_month").agg(orders=("order_id", "nunique"), gross_revenue=("revenue", "sum"))
              .reset_index())

def spark_monthly():
    o = read_csv("orders").filter(F.col("order_status") == "delivered")
    i = read_csv("order_items")
    return (o.join(i, "order_id")
             .withColumn("purchase_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
             .groupBy("purchase_month")
             .agg(F.countDistinct("order_id").alias("orders"),
                  F.sum(F.col("price") + F.col("freight_value")).alias("gross_revenue"))
             .orderBy("purchase_month").toPandas())

p_res, s_res = pandas_monthly(), spark_monthly()
assert (p_res["purchase_month"].tolist() == s_res["purchase_month"].tolist()
        and (p_res["orders"].values == s_res["orders"].values).all()
        and abs(p_res["gross_revenue"].values - s_res["gross_revenue"].values).max() < 0.01), "engines disagree"
print("Both engines return identical monthly results.\n")

t_pandas = min(best_of(pandas_monthly))
t_spark = min(best_of(spark_monthly))
bench = pd.DataFrame({
    "engine": ["pandas", "PySpark local[*]"],
    "best_of_3_seconds": [t_pandas, t_spark],
})
bench["relative_to_pandas"] = bench["best_of_3_seconds"] / t_pandas
display(bench)
print(f"Spark one-off JVM start-up (Step 1): {SPARK_STARTUP_S:.1f}s  |  machine cores visible to Spark: {spark.sparkContext.defaultParallelism}")

Both engines return identical monthly results.



,engine,best_of_3_seconds,relative_to_pandas
0,pandas,1.09,1.00
1,PySpark local[*],0.97,0.90


Spark one-off JVM start-up (Step 1): 10.5s  |  machine cores visible to Spark: 4


**📌 Observation:** Both engines return identical monthly results, so the timing comparison is fair. Read the two timings above: at this data size PySpark shows **no meaningful advantage over pandas** — the two land in the same range, and which one edges ahead changes with the machine and core count — and that is before counting Spark's one-off JVM start-up (the start-up figure reads ~0s if the Spark session was already running; restart the kernel for a true cold-start number).

**💡 Business insight:** This is the point of the honest note below — knowing *when not* to reach for Spark is part of the skill.

---
## 🧾 Honest note — what this notebook is, and is not

**This is an API-fluency demo; the data is small.** Olist is ~100K orders. At that size a single machine with pandas (or the SQLite layer already in this repo) does the job with less code and less overhead — Step 6 measured it. Spark's costs (JVM start-up, task scheduling, shuffles) only pay off when the work no longer fits one machine.

**When Spark *is* the right tool**
- data larger than one machine's memory/disk, or too slow to process on one machine;
- data already living in distributed storage (S3 / HDFS / a lakehouse) in Parquet or Delta;
- jobs that must scale out across a cluster, or streaming/incremental pipelines on the same API.

**What carries over:** the code above — explicit schemas, order-grain aggregation before joins, `broadcast` for small dimensions, window functions, `cache()`, partitioned Parquet, reading `explain()` — is the same on a 100-node cluster. Only the `master` and the configuration change.

**What this notebook deliberately does *not* claim:** no cluster tuning, no skew handling, no benchmarks beyond the single local measurement above (which depends on this machine). KPI results match the SQL layer in `sql/`; the SQL layer remains the project's source of truth.